In [ ]:
!nvidia-smi

# Qwen 2.5 1.5B: cross-domain transfer

Model: Qwen/Qwen2.5-1.5B | [HF](https://huggingface.co/Qwen/Qwen2.5-1.5B)\
GPU: A100 40GB | WikiText → C4/Code transfer | Date: 2026-04-06

In [ ]:
!git clone https://github.com/tmcarmichael/nn-observability.git
%cd nn-observability
!pip install uv huggingface_hub -q
!uv sync --extra transformer -q

try:
    from google.colab import userdata
    import os
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("Not on Colab or no HF_TOKEN secret")

In [ ]:
# Pre-download models (uses HF_TOKEN from env)
!hf download Qwen/Qwen2.5-1.5B


In [ ]:
import os
os.environ.pop("MPLBACKEND", None)

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_observe import collect_layer_data, load_wikitext, train_linear_binary, evaluate_head, bootstrap_ci
from observe import partial_spearman

In [ ]:
# Load Qwen 2.5 1.5B
model_id = 'Qwen/Qwen2.5-1.5B'
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa"
).cuda()
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

n_layers = model.config.num_hidden_layers
hidden_dim = model.config.hidden_size
print(f'{n_layers} layers, {hidden_dim} dim')

peak_layer = 19

# Sanity check
with torch.inference_mode():
    test_ids = tokenizer("test", return_tensors="pt")["input_ids"].cuda()
    test_out = model(test_ids, output_hidden_states=True)
    assert len(test_out.hidden_states) == n_layers + 1
    print(f'Sanity check passed: {len(test_out.hidden_states)} hidden states')

In [ ]:
# Load WikiText train, train observer
train_docs = load_wikitext('train', max_docs=2000)
print(f'{len(train_docs)} train docs')

# Token budget: 150 ex/dim
max_train = max(150 * hidden_dim, 200000)
print(f'Token budget: {max_train} ({max_train/hidden_dim:.0f} ex/dim)')

print('Collecting WikiText train activations at peak layer...')
train_data = collect_layer_data(model, tokenizer, train_docs, peak_layer, 'cuda', max_train)

# Train observer heads
heads = []
for seed in [42, 43, 44]:
    heads.append(train_linear_binary(train_data, seed=seed))
print(f'Trained {len(heads)} observer heads')

In [ ]:
# Define domain loaders
from datasets import load_dataset

def load_c4_docs(split='test', max_docs=500):
    """Load C4 (cleaned web text) documents."""
    ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
    docs = []
    skip = 50000 if split == 'test' else 0
    for i, row in enumerate(ds):
        if i < skip:
            continue
        text = row['text'].strip()
        if len(text) > 100:
            docs.append(text)
        if max_docs and len(docs) >= max_docs:
            break
    return docs

def load_code_docs(split='test', max_docs=500):
    """Load CodeSearchNet Python documents."""
    ds = load_dataset('code_search_net', 'python', split='test', streaming=True, trust_remote_code=True)
    docs = []
    for row in ds:
        text = row['whole_func_string'].strip()
        if len(text) > 100:
            docs.append(text)
        if max_docs and len(docs) >= max_docs:
            break
    return docs

In [ ]:
# Evaluate on each domain
max_test = max(75 * hidden_dim, 100000)

domains = {
    'wikitext': load_wikitext('test', max_docs=500),
    'c4': load_c4_docs('test', max_docs=500),
    'code': load_code_docs('test', max_docs=500),
}

results = {}
for domain_name, docs in domains.items():
    print(f'\n--- {domain_name} ({len(docs)} docs) ---')
    test_data = collect_layer_data(model, tokenizer, docs, peak_layer, 'cuda', max_test)
    
    rhos = []
    for seed, head in zip([42, 43, 44], heads):
        _, rho, p = evaluate_head(head, test_data)
        rhos.append(rho)
        print(f'  seed {seed}: partial corr = {rho:+.4f}')
    
    mean_rho = float(np.mean(rhos))
    results[domain_name] = mean_rho
    print(f'  mean: {mean_rho:+.4f}')

print('\n=== CROSS-DOMAIN TRANSFER (Qwen 1.5B, layer 19) ===')
print(f'{"Domain":<15} {"Partial corr":>14} {"Transfer ratio":>16}')
print('-' * 47)
ref = results['wikitext']
for name, rho in results.items():
    ratio = '(reference)' if name == 'wikitext' else f'{rho/ref:.2f}'
    print(f'{name:<15} {rho:>+14.4f} {ratio:>16}')

In [ ]:
print('\n--- Within-domain C4 test ---')
c4_train = load_c4_docs('train', max_docs=2000)
print(f'Loaded {len(c4_train)} C4 train docs')
c4_train_data = collect_layer_data(model, tokenizer, c4_train, peak_layer, 'cuda', max_train)

c4_test_data = collect_layer_data(model, tokenizer, domains['c4'], peak_layer, 'cuda', max_test)

c4_rhos = []
for seed in [42, 43, 44]:
    c4_head = train_linear_binary(c4_train_data, seed=seed)
    _, rho, p = evaluate_head(c4_head, c4_test_data)
    c4_rhos.append(rho)
    print(f'  seed {seed}: partial corr = {rho:+.4f}')
print(f'  Within-domain C4: {np.mean(c4_rhos):+.4f}')
print(f'  Cross-domain C4 (WikiText-trained): {results["c4"]:+.4f}')

In [ ]:
# Save results
import json
output = {
    'model': 'Qwen/Qwen2.5-1.5B',
    'peak_layer': peak_layer,
    'cross_domain': {name: float(rho) for name, rho in results.items()},
    'within_domain_c4': float(np.mean(c4_rhos)),
}
with open('cross_domain_qwen.json', 'w') as f:
    json.dump(output, f, indent=2)
print(json.dumps(output, indent=2))

from google.colab import files
files.download('cross_domain_qwen.json')